### Imports and Paths


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from IPython.display import display

# Paths relative to the repository root (the folder containing this notebook).
PRODUCTS_PATH = Path("part1_processed_products.json")
VALIDATION_PATH = Path("inputs") / "validation_labels.csv"

# If this notebook is stored in a subfolder, adjust the paths above accordingly.
try:
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    # The Part 1 environment normally has this corpus. If it is absent and
    # downloading is unavailable, use scikit-learn's built-in English list.
    try:
        nltk.download("stopwords", quiet=True)
        STOP_WORDS = set(stopwords.words("english"))
    except Exception:
        from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
        STOP_WORDS = set(ENGLISH_STOP_WORDS)
        warnings.warn(
            "NLTK stopwords could not be loaded; using scikit-learn's English "
            "stop-word list. For exact Part 1 consistency, install NLTK stopwords."
        )
STEMMER = PorterStemmer()

print("Working directory:", Path.cwd())
print("Products file exists:", PRODUCTS_PATH.exists(), PRODUCTS_PATH)
print("Validation file exists:", VALIDATION_PATH.exists(), VALIDATION_PATH)


### Load and validate product collection

In [ ]:
if not PRODUCTS_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {PRODUCTS_PATH}. Put the file in the project root "
        "or update PRODUCTS_PATH."
    )

with PRODUCTS_PATH.open("r", encoding="utf-8") as f:
    products = json.load(f)

if not isinstance(products, list):
    raise TypeError("Expected part1_processed_products.json to contain a list of product records.")

products_df = pd.DataFrame(products)

required_fields = [
    "pid", "title", "description", "brand", "category", "sub_category",
    "product_details", "seller", "out_of_stock", "selling_price",
    "discount", "actual_price", "average_rating", "url"
]
missing_fields = [field for field in required_fields if field not in products_df.columns]
if missing_fields:
    raise ValueError(f"Missing required product fields: {missing_fields}")

products_df = products_df.dropna(subset=["pid"]).copy()
products_df["pid"] = products_df["pid"].astype(str)
if products_df["pid"].duplicated().any():
    raise ValueError("Product IDs (pid) must be unique to build an unambiguous index.")

print(f"Loaded {len(products_df):,} products")
print(f"Unique product IDs: {products_df['pid'].nunique():,}")
display(products_df[["pid", "title", "category", "sub_category"]].head())


### Preprocessing

In [ ]:
def safe_text(value):
    """Return an empty string for missing values; otherwise return text."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value)


def flatten_product_details(details):
    """Flatten product_details list/dict key-value pairs into searchable text."""
    parts = []
    if isinstance(details, list):
        for item in details:
            if isinstance(item, dict):
                for key, value in item.items():
                    if value is not None and not pd.isna(value):
                        parts.extend([safe_text(key), safe_text(value)])
    elif isinstance(details, dict):
        for key, value in details.items():
            if value is not None and not pd.isna(value):
                parts.extend([safe_text(key), safe_text(value)])
    elif details is not None and not pd.isna(details):
        parts.append(safe_text(details))
    return " ".join(parts)


def preprocess_text(text):
    """Lowercase, tokenize, remove English stop words and Porter-stem tokens."""
    text = safe_text(text).lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return [
        STEMMER.stem(token)
        for token in tokens
        if token not in STOP_WORDS and len(token) > 1
    ]


TEXT_FIELDS = [
    "title", "description", "category", "sub_category",
    "brand", "seller"
]

def product_to_text(product):
    """Combine searchable text fields and flattened product details for one product."""
    values = [safe_text(product.get(field, "")) for field in TEXT_FIELDS]
    values.append(flatten_product_details(product.get("product_details", [])))
    return " ".join(value for value in values if value)


products_by_pid = {
    str(product["pid"]): product
    for product in products
    if product.get("pid") is not None
}

# Tokenize each product once; the same preprocessing is used for queries.
document_tokens = {}
for pid, product in products_by_pid.items():
    document_tokens[pid] = preprocess_text(product_to_text(product))

print("Example original title:", products[0].get("title"))
print("Example processed tokens:", document_tokens[str(products[0]["pid"])][:30])


### Inverted Index

In [ ]:
def build_inverted_index(document_tokens):
    """Build term -> {pid: [positions]} inverted index and document lengths."""
    index = defaultdict(dict)
    document_lengths = {}

    for pid, tokens in document_tokens.items():
        document_lengths[pid] = len(tokens)
        positions_by_term = defaultdict(list)

        for position, term in enumerate(tokens):
            positions_by_term[term].append(position)

        for term, positions in positions_by_term.items():
            index[term][pid] = positions

    return dict(index), document_lengths


inverted_index, document_lengths = build_inverted_index(document_tokens)

num_documents = len(document_tokens)
vocabulary_size = len(inverted_index)

print(f"Documents indexed: {num_documents:,}")
print(f"Vocabulary size: {vocabulary_size:,}")
print("Example vocabulary entries:")
for term in list(inverted_index)[:5]:
    print(term, list(inverted_index[term].items())[:3])


### Inspect vocabulary and test queries

In [ ]:
term_frequency = Counter()
document_frequency = {
    term: len(postings)
    for term, postings in inverted_index.items()
}
for term, postings in inverted_index.items():
    term_frequency[term] = sum(len(positions) for positions in postings.values())

vocabulary_stats = pd.DataFrame([
    {
        "term": term,
        "term_frequency": term_frequency[term],
        "document_frequency": document_frequency[term]
    }
    for term in inverted_index
]).sort_values(
    ["term_frequency", "document_frequency"],
    ascending=False
)

display(vocabulary_stats.head(30))


In [ ]:
test_queries = [
    "women cotton kurta",
    "men slim fit jeans",
    "blue printed dress",
    "leather casual shoes",
    "women running shoes"
]

for query in test_queries:
    print(
        f"{query!r} -> processed: {preprocess_text(query)}"
    )


### Conjunctive (AND) retrieval

In [ ]:
def search_and(query, index=inverted_index):
    """Return PIDs containing every distinct processed query term (AND semantics)."""
    query_terms = list(dict.fromkeys(preprocess_text(query)))
    if not query_terms:
        return []

    # A missing query term makes an AND query impossible to satisfy.
    if any(term not in index for term in query_terms):
        return []

    # Start with the shortest posting list to reduce intersection work.
    query_terms.sort(key=lambda term: len(index[term]))
    candidates = set(index[query_terms[0]])

    for term in query_terms[1:]:
        candidates.intersection_update(index[term])
        if not candidates:
            break

    return sorted(candidates)


for query in test_queries:
    matches = search_and(query)
    print(f"{query!r}: {len(matches):,} AND matches")


### ranking of AND-matches

In [ ]:
idf = {
    term: math.log(1 + num_documents / (1 + len(postings)))
    for term, postings in inverted_index.items()
}

# Precompute the full TF-IDF L2 norm for every document. The norm must include
# every indexed term, not only the terms shared with a particular query.
document_tfidf_norms = {}
for pid, tokens in document_tokens.items():
    length = len(tokens)
    if length == 0:
        document_tfidf_norms[pid] = 0.0
        continue
    counts = Counter(tokens)
    document_tfidf_norms[pid] = math.sqrt(sum(
        ((count / length) * idf[term]) ** 2
        for term, count in counts.items()
    ))


def tfidf_rank(query, candidate_pids, index=inverted_index):
    """Rank AND-matched product IDs by cosine similarity of TF-IDF vectors."""
    query_tokens = preprocess_text(query)
    if not query_tokens or not candidate_pids:
        return []

    query_counts = Counter(query_tokens)
    query_weights = {
        term: (count / len(query_tokens)) * idf.get(term, 0.0)
        for term, count in query_counts.items()
    }
    query_norm = math.sqrt(sum(weight ** 2 for weight in query_weights.values()))
    if query_norm == 0:
        return []
    query_weights = {
        term: weight / query_norm
        for term, weight in query_weights.items()
    }

    ranked = []
    for pid in candidate_pids:
        doc_length = document_lengths.get(pid, 0)
        doc_norm = document_tfidf_norms.get(pid, 0.0)
        if doc_length == 0 or doc_norm == 0:
            continue

        # Only shared terms contribute to the dot product, but the document
        # norm above is computed over the complete document vector.
        score = 0.0
        for term, query_weight in query_weights.items():
            positions = index.get(term, {}).get(pid)
            if positions:
                tf = len(positions) / doc_length
                doc_weight = tf * idf[term]
                score += query_weight * doc_weight / doc_norm
        ranked.append((pid, float(score)))

    ranked.sort(key=lambda item: (-item[1], item[0]))
    return ranked


def search(query):
    """Run AND retrieval followed by TF-IDF ranking; return (pid, score) pairs."""
    candidates = search_and(query)
    return tfidf_rank(query, candidates)


def show_results(query, k=10):
    """Display the top-k ranked products and required product metadata."""
    ranked = search(query)
    rows = []
    for rank, (pid, score) in enumerate(ranked[:k], start=1):
        product = products_by_pid[pid]
        rows.append({
            "rank": rank,
            "pid": pid,
            "score": round(score, 6),
            "title": product.get("title", ""),
            "brand": product.get("brand", ""),
            "category": product.get("category", ""),
            "sub_category": product.get("sub_category", ""),
            "selling_price": product.get("selling_price", ""),
            "average_rating": product.get("average_rating", ""),
            "url": product.get("url", "")
        })
    return pd.DataFrame(rows)


show_results("women full sleeve sweatshirt cotton", k=10)

### Evaluation of queries

In [ ]:
if not VALIDATION_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {VALIDATION_PATH}. Put validation_labels.csv in inputs/ "
        "or update VALIDATION_PATH."
    )

validation = pd.read_csv(VALIDATION_PATH)
expected_columns = {"title", "pid", "query_id", "labels"}
if not expected_columns.issubset(validation.columns):
    raise ValueError(
        f"validation_labels.csv must contain {sorted(expected_columns)}; "
        f"found {validation.columns.tolist()}"
    )

validation["pid"] = validation["pid"].astype(str)
validation["query_id"] = pd.to_numeric(validation["query_id"], errors="raise").astype(int)
validation["labels"] = pd.to_numeric(validation["labels"], errors="raise").astype(int)

if not set(validation["labels"].unique()).issubset({0, 1}):
    raise ValueError("Expected binary labels 0 and 1.")

predefined_queries = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue"
}

print("Judgements per query:")
display(validation.groupby("query_id")["labels"].agg(
    judgement_count="count",
    relevant_count="sum"
).reset_index())


In [ ]:
def precision_at_k(ranked_pids, relevant_pids, k):
    """Fraction of the first k retrieved products that are relevant."""
    if k <= 0:
        raise ValueError("k must be positive")
    top_k = ranked_pids[:k]
    return sum(pid in relevant_pids for pid in top_k) / k


def recall_at_k(ranked_pids, relevant_pids, k):
    """Fraction of all judged relevant products retrieved in the first k."""
    if not relevant_pids:
        return 0.0
    top_k = ranked_pids[:k]
    return sum(pid in relevant_pids for pid in top_k) / len(relevant_pids)


def average_precision_at_k(ranked_pids, relevant_pids, k):
    """Compute AP@k, normalized by min(number of relevant products, k)."""
    if not relevant_pids or k <= 0:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for rank, pid in enumerate(ranked_pids[:k], start=1):
        if pid in relevant_pids:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / min(len(relevant_pids), k)


def f1_at_k(ranked_pids, relevant_pids, k):
    """Harmonic mean of precision@k and recall@k."""
    p = precision_at_k(ranked_pids, relevant_pids, k)
    r = recall_at_k(ranked_pids, relevant_pids, k)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def reciprocal_rank(ranked_pids, relevant_pids):
    """Return reciprocal rank of the first relevant result, or zero if none."""
    for rank, pid in enumerate(ranked_pids, start=1):
        if pid in relevant_pids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(ranked_pids, relevant_pids, k):
    """Compute binary-relevance NDCG@k."""
    if k <= 0:
        raise ValueError("k must be positive")
    gains = [1 if pid in relevant_pids else 0 for pid in ranked_pids[:k]]
    dcg = sum(gain / math.log2(rank + 1)
              for rank, gain in enumerate(gains, start=1))
    ideal_gains = [1] * min(len(relevant_pids), k)
    idcg = sum(gain / math.log2(rank + 1)
               for rank, gain in enumerate(ideal_gains, start=1))
    return 0.0 if idcg == 0 else dcg / idcg


def evaluate_one_query(ranked_pids, relevant_pids, k=10):
    """Return all requested per-query metrics at cutoff k plus reciprocal rank."""
    return {
        "P@K": precision_at_k(ranked_pids, relevant_pids, k),
        "R@K": recall_at_k(ranked_pids, relevant_pids, k),
        "AP@K": average_precision_at_k(ranked_pids, relevant_pids, k),
        "F1@K": f1_at_k(ranked_pids, relevant_pids, k),
        "RR": reciprocal_rank(ranked_pids, relevant_pids),
        "NDCG@K": ndcg_at_k(ranked_pids, relevant_pids, k)
    }


In [ ]:
K = 10

ranked_by_query = {}
relevant_by_query = {}
per_query_metrics = []

for query_id, query_text in predefined_queries.items():
    qrels = validation[validation["query_id"] == query_id]
    relevant_pids = set(qrels.loc[qrels["labels"] == 1, "pid"])
    ranked_pids = [pid for pid, score in search(query_text)]

    ranked_by_query[query_id] = ranked_pids
    relevant_by_query[query_id] = relevant_pids

    metrics = evaluate_one_query(ranked_pids, relevant_pids, K)
    metrics["query_id"] = query_id
    per_query_metrics.append(metrics)

metrics_df = pd.DataFrame(per_query_metrics).set_index("query_id")
# Assignment asks for numeric values rounded to three decimal places.
display(metrics_df[["P@K", "R@K", "AP@K", "F1@K", "RR", "NDCG@K"]].round(3))


### Validation coverage and possible limitations

In [ ]:
known_pids = set(products_by_pid)
validation_pids = set(validation["pid"])
missing_validation_pids = sorted(validation_pids - known_pids)

print("Validation PIDs:", len(validation_pids))
print("Validation PIDs found in product collection:", len(validation_pids & known_pids))
print("Validation PIDs missing from product collection:", len(missing_validation_pids))
if missing_validation_pids:
    print("Missing examples:", missing_validation_pids[:10])


### Custom test queries

In [ ]:
candidate_rows = []
for query in test_queries:
    for rank, (pid, score) in enumerate(search(query)[:30], start=1):
        product = products_by_pid[pid]
        candidate_rows.append({
            "query": query,
            "rank": rank,
            "pid": pid,
            "score": score,
            "title": product.get("title", ""),
            "description": product.get("description", ""),
            "category": product.get("category", ""),
            "sub_category": product.get("sub_category", ""),
            "brand": product.get("brand", ""),
            "product_details": flatten_product_details(product.get("product_details", [])),
            "relevance": ""  # Fill manually with 1 or 0 after reviewing the product.
        })

candidate_judgements_template = pd.DataFrame(candidate_rows)
custom_judgements_path = Path("inputs") / "custom_query_judgements_template.csv"
custom_judgements_path.parent.mkdir(parents=True, exist_ok=True)
candidate_judgements_template.to_csv(custom_judgements_path, index=False)
print("Saved manual judgement template:", custom_judgements_path)
display(candidate_judgements_template.head(10))


In [ ]:
CUSTOM_JUDGEMENTS_PATH = Path("inputs") / "custom_query_judgements.csv"

if CUSTOM_JUDGEMENTS_PATH.exists():
    custom_qrels = pd.read_csv(CUSTOM_JUDGEMENTS_PATH)
    required_custom = {"query", "pid", "relevance"}
    if not required_custom.issubset(custom_qrels.columns):
        raise ValueError(f"Custom judgements need columns: {sorted(required_custom)}")

    custom_qrels["pid"] = custom_qrels["pid"].astype(str)
    custom_qrels["relevance"] = pd.to_numeric(
        custom_qrels["relevance"], errors="coerce"
    )
    if custom_qrels["relevance"].isna().any():
        raise ValueError("Complete every relevance cell with 0 or 1 before evaluation.")
    if not set(custom_qrels["relevance"].unique()).issubset({0, 1}):
        raise ValueError("Custom relevance labels must be binary 0 or 1.")

    custom_rows = []
    custom_ranked = {}
    custom_relevant = {}

    for query in test_queries:
        qrels = custom_qrels[custom_qrels["query"] == query]
        relevant = set(qrels.loc[qrels["relevance"] == 1, "pid"])
        ranked = [pid for pid, score in search(query)]
        custom_ranked[query] = ranked
        custom_relevant[query] = relevant

        row = evaluate_one_query(ranked, relevant, K)
        row["query"] = query
        custom_rows.append(row)

    custom_metrics_df = pd.DataFrame(custom_rows).set_index("query")
    display(custom_metrics_df[["P@K", "R@K", "AP@K", "F1@K", "RR", "NDCG@K"]].round(3))

    custom_map = np.mean([
        average_precision_at_k(custom_ranked[q], custom_relevant[q], K)
        for q in test_queries
    ])
    custom_mrr = np.mean([
        reciprocal_rank(custom_ranked[q], custom_relevant[q])
        for q in test_queries
    ])
    display(pd.DataFrame([{"MAP": custom_map, "MRR": custom_mrr}]).round(3))
else:
    print(
        "Manual judgements are not complete yet. Fill the relevance column in "
        f"{custom_judgements_path} and save it as {CUSTOM_JUDGEMENTS_PATH}."
    )


### Export index

In [ ]:
# Optional compact index export: term -> PID -> positions
EXPORT_INDEX = False

if EXPORT_INDEX:
    index_json = {
        term: {pid: positions for pid, positions in postings.items()}
        for term, postings in inverted_index.items()
    }
    with open("inverted_index_part2.json", "w", encoding="utf-8") as f:
        json.dump(index_json, f, ensure_ascii=False)
    print("Saved inverted_index_part2.json")

print("Index statistics:")
print(f"Documents: {num_documents:,}")
print(f"Vocabulary terms: {vocabulary_size:,}")
print(f"Average unique terms per product: {np.mean([len(set(t)) for t in document_tokens.values()]):.2f}")


Suggested topics of discussion for report:
- Precision vs recall: how many top results are relevant, versus how many judged relevant products are retrieved?
- AP/MAP: whether relevant products tend to appear early across queries.
- MRR: how soon the first relevant result appears.
- NDCG: whether relevant products are concentrated near the top.
- AND-query limitation: a product missing one query term is excluded even if it is otherwise relevant.
- Lexical matching: stemming helps with word variants but does not solve synonyms, spelling errors or semantic mismatch.
- Field handling: concatenating fields is a simple baseline; title and structured product details may deserve stronger weights than long descriptions.
- Judgement coverage: validation labels cover only a finite set of products, so unjudged products may be treated as non-relevant by this evaluation.

Ensure reproductibility